# 14 · A glimpse of more: buoyant convection ☕🔥

In notebooks 9 and 11 the coffee cooled passively. But a real cup *moves*: warm
fluid is lighter, so it **rises**, cool fluid **sinks**, and a circulation sets
in. We couple the **temperature** $T$ to a **flow** $\mathbf u$ through the
**Boussinesq** approximation — buoyancy enters the momentum balance as a force
proportional to temperature:
$$
-\Delta\mathbf u + \nabla p = \mathrm{Ra}\,T\,\hat{\mathbf y},\quad
\nabla\!\cdot\mathbf u = 0,\qquad
\partial_t T + \mathbf u\!\cdot\!\nabla T = \Delta T .
$$

The trick that keeps this **cheap and portable** (only the symmetric, positive
`sparsecholesky` solver — no indefinite or non-symmetric direct solver needed) is
the one the NGSolve i-tutorials use: march in **time** with an **IMEX** scheme —
treat the troublesome **convection explicitly** — and replace the Stokes
saddle-point by a **grad–div penalty**, so *every* solve is symmetric positive
definite.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from IPython.display import display, HTML

_BEAST = "iVBORw0KGgoAAAANSUhEUgAAAFQAAABUCAYAAAAcaxDBAAAeCklEQVR42u2c6bNlV3mff+/a83Dm4Z47d98epdbUQgIhCWGMsJFHDMZRwE4qnrCJh8R4IOU4ZcDgKuPCsUVMiMGOq8AmGKcYDZgYbJCEJiSkVqvV6unevsOZh33O2fPe682H5EO+uCqpbtQQ3+cvWL+n3r33WrXfdwH77LPPPvvss88+++yzzz777LPPPv+EoWu9gP8b/vD1gATQEcArM+D7PnmtV/SP820n9PD3vBcsGZqa4ditHdxlP4yj+Ryb9UNoYwnWvA9jNoUXWugrdXT7i+jcsYwn3vKL13rpAAD1Wi/grvf9Cp48sIH3fvqDSK8v4xFNxcd/tyOWj7fsXFMrQsqSwlJnzhXOUmIWGYBM5XxGxCMaZv7W127K7v3Zd+NG5xEoXzoF/VAZ7/nUN/9pCb39de/E3BOo7fXxn5/5fYS5bebhfBVZ/8a7f7B100qzd/Tu7PFW2ferpxZv0Z7on1TiKVBd8PjWxiP5iTNnJ93SpPv1u5qXbr74+DOuO3xaT4LN40++yvvGvzoDMOPzRLjv/2eh7wYQf7eNhVmAvyl/Fb/yW/8Df/aJ32qcXzh0x+H25XvH5vGXzsSB1cWjU/e76BvqRvsSn26dkI9Hd3J/Vkertk3Lelup7M45zqy1tFqkyKlIPzDDbF7p7h09/Iz8hee/dClZ/uz9P/49o/s+xLjj0V/HI3/y3hct44v2Dr3uX78ab3707zColPGpC0dx/Q9U6v3zB15j1aI3HtGff4lM9dKWcZICt4yXxY8rC9mIto8d5Z3xS2DaMW4yHsd13dOQKXinuEwv2MexyYdZngfnW21oaw4fuf0Crr/0ZOJdmH7CT8N3bT+j7f3ZpV0cvf+9eOFjv/ai5FReLKFa+43Y0RbxkuUn1ZY3vCdN0//QV8VbBgPlxN7Msvs9Uv3OhMK8L+TOBdHux/T0uEU7bZVK5lN8fPNpEkpOzxy8WTyJe6iXHWYdIRWcDpmrCZrHJ7hBOU+LmwMHVXHL37m3177Ye+Wjrz3C/kOzP8U7Lr84OV8UoZUfezfcxQ483y3tlK2f17T0t3U/fqkZ+pZTmMEyIiE1BXHFEK4xFYveZeH2OuBajO7UkcHARzCc0BlRoCcmxxFsa7y4eo7uG36Clk8/StclW3RAjElNISRYTZct/Qlxw/WUsH5o9fQjf1a+Pr7zcAnffK77nS30P35XEdcdMoBRjh3lYMUYD96OWPulme00wkqJbDXC2mxPNJM+NCWFG4e0FmyjNJmQKOrYrS3T0C+RThHVxS6cTpdmyGhvkMLrR4qeBnApJGq4tLmwgSf1o7xpVJSFZCiqc1+TOd1gbfnT7zv38BPtoCAnN/wqxhf+7jtX6O3fXUKmKbixeNbmevA26WpvZQh7HjUwiytKppuISypnQofWTYQxm5NrxCJuunRp5SBtjY6LvO1DzMakmJqoKDM0xYSyouDh1KXnvAafNVbFafUW2tpZp/DZnHqbmtIvl8Sx4Tkqz/vGbvPo9Q8u3HHKXKlfOB85WHj5D2H0zS995wn9rt/9EayNB/g3kz36+p7xZns++Y2C6hfUMsOqJ2SIjEO/gH66QW2tKbioUrqhCs9xaIcPohMc4WxqQRUANIOIDUrcqjDKOVZkH01nIoSd0yxReDpOKIk9mIWcOFWUIG4KazXD8uZ5+K1aaSu/tXXxwqGvtJo8PfaaF/Cm14T48ucG31lCb/yDg5D/0MFnr9u4TSd+Tz5R1tDP88V5T6z62+QUYooXNUWSiWxUxnReoyA3OB1VKJ5VhCwrXCq1sWxsU6kZc6irSGYuDdwKuaUZLfc7vDjpUkvpoeIMsCC2qBKfJcXJhFc/IPyqgeWkg7KTYrt4ZD2rFXnlB5976Oj5rfRlD27hL8743zlCj93+VvSfWMFF+fLKbFp4R9es39Ovl9hpAqYZiSjXSBuk1Ar2RLOwx3ETNB5a4I7HTpay0cqxWLsoDnYvC3drV9jDrqKUEmXkVpVICoLjkWqnyKEII07RiGYoTmYkdjwo07YY1RUxKNdxQt1FY9LDdrEmtCi/4WB+evRrT3/pqVOaLV9vZvjkXv7tL7T1R++CeibCdvaDVGt/5U3K3uDnZwNLD9IG+dxShs0mRq0yFWqpaIzHsnjZg+pITCrLyGYa6U4mqpWLYqHTIbkbEguQZoIKekZ6bU5V0aWF3khEpqE8ZRwXu0qT5nZRpEUTVDV4WiyLdtoUcUS4Xt1GfW+AWcHFeLhkLsvdG9Hrn3vzfxudvfcvHkDp7a/Hs5995qrmv+onpVcvfACnSq/DGr5+xPDkT8k4K+aaJi0/QRQ6NBnWwXrO8/VVjg4+g4PKJS53YjSXdjEuZrw03SS14/OOUkEgdMocE6ojYUcZNQZzpeINBbyYQrOEIGgi20qFZ7dglXyhOnkeSQdJ14WVxNgJAqzmDM0C5nYBM624OrGM3/zzn1rcGivi6VtED395lfNf9Qp1vv9e4P49DUP7rcLUfyRrVUX1sMTK0hj6aoQZVynrRMxehqhapHLNx/RyLvuZJhU9QJ4Q9/QF7AZLYuSpmM5szFCFKEQoRCMpRr5kW8Hl1oaYDRchfEBRUyCQJMcaZRMQJ4Kceoxmtg0R5ri8tIbL8w1IO0A97Dfb5dXSR171xq/ujMph0fwZ9F741Len0Le88w04FO7i5LODW1fV7r9fKE1qC47HJ/2n6Xj3LCr2iGRNwcwowuqMQVJD31HyS14LnuYKkYYCgkRgmvAzFxTngoQgvZqIWjKCOZtTVla4Z9d5ECwhNmrCPDhHZbkPvZ5BK5lERY1EQ2ClsouV/h7mvoqBs4jJ6AAUPcXaaId6Sm29M9gYPf6e33l8+ebH+N6fL+GZz5+6Kg6u2iN//J//V9TtP0ftoGIkn47eXN9qH1y1FE7tCqk0Y5rG1MqnrKyfQbahwdstIe53ZD8wYFNCG+p5lMMR5XlOI7YUuz7iQFQQexZZdsz6JEcam2JUdeUkqoPOe7LcfIaPFwZKvTskoShIay5GRQt5Smj1hzBHEabVCryZjTyUsPQY5KYk1MjVMuUtt//sf/q6shk/9uFf/Cg+cpU8XLUK7T/7SfzJq27HY72XvLQhBm+397rlpFpFZ2OVdo9VRbzsUItTVOcBlHKGLhn5aBKxk07Ecr5FBW1GUgUyoQKeoIKYUUYKZ54LRQdRJSHbT4QiJZKSxqypoFjyQjCn1nQsNGaonoTVjlDu+3CDCFQzcXn9AHYGB6GEHhYWuzgyPEOiLPIBliuLzgvG61be/+VHNirJj+oSn9nLrtjDVanQ1b/4Ddz8xy2IDxlGfqt2/7Nr0fLS3TVspuvUHlbIGOawjk65fuBrWL84wNJomGtJgVO9Jlb8HcqqKm2LDQRSkFaS1IoHEPMpF5ZCEUY5spFKk0KZjYWcy12P1KQv0iUFls2shnNuH29yWC6R4WfQZxGADJ5bwMhZwqXJESS7Pmw7RXHqwZglUGYQWZKxFbbv4676yf6ra5+uPTa6KoV1VYRuH1vHoeIE7dvzm62O9b3euaYYZoRJL2AEA6QbG0KcSMBkY1Kr4fyOSu12nSwOheMmclQsUbBtI5jHFK8scNmekbVNZBRzNloB4p0iaM+Ct16D1UigbwVkpcRVI4QWhfKydQhb4g5V12dQV0YgO0I2LiDZKyHsERCHEC0LmjpDnEuMLaJAIvdVq56a+o8vPLL31b2bm5P6z7wbg7f95LUX+sCf/hEOJ9va3zdvfV1nubUyHqzmncsNhbKIzOsKWNmY8MnRKZQy5m84i7znB+wrlii5E9jzgHIBnokCKUEklSChaa0OuzRjZSdD/cCEslVQfKnK8W6Rd1YIpUMzaHmCoFSCyVkeRiXIaY3jwKY4VqAaCjRhQXMJcpWQrFRgrO2gyFNkjomuVmVvbslRpY6BU73LrzfvOLHzjS9Eq+VrX6E33PcAWt13IKvh0F0PPfW9kasrOwdW5OUbj2aJ5VAr7dJCd5NEkOLpjZM43a9w1fomLI5RVaZcHntCo4S69QYbQ02q7R4is0mjjTqaF/qwdwNuHmaMl1yeXVApPiewVznMrAqIuU5OmipWGmLtplOYDhYRtiuAGWBx4zzqRh9JqCG0dZQKUyxt++CiholWpHzMQig5T0Onfl5b/oE/0N7ylbVPPx3Pr7XQrFrGj35kgJ9791vuXpfnD5QnfVlsM+4YPkmuFiL3c5q6Ds7Ub+Lz5w+j2j3Ny0kHU7/GigXKDWItzqAsZ4gbVdDWiLKLEfeP1FlZlbywtQf7ciLSdRuj4grnuyMwSsjMKrHFMAs5FSYj3NK7jM7RNWyuHUOptItD7QtQH5tA9QXsVRv6mgEnytAzdMRTA3o2JNICZJNU+BNx96HLZ9ZUNzv3ig89iK/99N3XTqjBQ9z4xveZRkF/yTPuCVMYmWyuDcWitcOmnFPPaEieNdXhdhHBeIB6NiZKMg6TjFK7QaP1ORXaAYpRzr0lRqYu5HKmMnd10WmRrLZ6ZF3ypDP3kJc3KB9VSLVnqB70uMgxLehzLAuP1gZTqi/toH7EhzNhFPspxrqFpKHBWypCK2pQohh5BmAccyHqcVmbsuGmDNVYFxrdIuPoXKDedW0r9LbKV5CmqVOrFVfmWkt2wnoOT8CbLFKnqcMhIXRnkt966DRMf6ZO9oZo++VsNtcR9xygvJIft8/w0rxLpjlB5Go0LbiiEx6mIFsgv7SFiuUx8gi6wwiaq1xsbPEybVPz7BRqUQPfuqCg5aAUEcgDLjglnDp2COFBG2Huwhcl1AuXcXwUQp17cDUfWeqwPZ3BTUcw3KFtt+xb7v7ruz9x6l2/ytdUqKM+j5RTtZQWLFsf5ctujJo6g28WxdBvSHOasGKqRIZglgbgLiqTyJbxDDKOZuynAJmUmLGcNvveJNjxx27NzJO1iruTHqrnql01wI7kDLoGzKwCFU3m1eEEGRO61q2i3zwk6uk3sd71kFgKzquHcLl/A4zNPrRRG4lJ2Ln1AMr+JspdH9ODGvySRQkLpnnCDnmiqorj0e+9v7CeqdNrKvRytoicQVnoK2yOqfb0ruKFkTh98mbl7HhVTbsaQUhWCiUyhS7UaYcyoekr122n2B3ukBF90UL6ICX5eZklY38n9pQszrQDsSO0rKbnwVElyF7FDeVelsaqyDRFWIIQlrljHRQ78xOKGbtIiwRxdgons7FcmCMoj5GbElmSw5l74E1GWMhQnOdwpiTb5jJG7pjTTAelkp0kWHT6Q1e7QqFXfFJqbpyElxTd0mz+hhq6S6lBQnhMTidkuzAkr5LLuacguzTltN9Fmmcil6CiGosD0Ti9XnC7eWixf/2a5b3WDSevPczxK3/WCbNfun2etD/cPf1A8XnXSb66dUv9dDYrHTpA09XWyZD2ancrw+6KQgxa2BBYq13AwlYH5Z6P1VkXK+ISnNUYs6NLyOcu5HiMam0Iaz7h1DbyvtdgoSVoql0xFxXKYi1YVoafTBJ18PhT/WtXoZkK5Fou52xmxdiVp2rHxbLeo+WzF0Tr1Bk4re38cmmFt5Q1hJuaau2OMKdCFlqGGJoLjVORer9xyXx9szqY3lgQg0LR6QwvbbQ7H132k8nH4rX7RxyUX6ofumXivCL9RvXATleMGpZ4SIswyE1M20Uk4wIuuieBmwwsTLpo9IY4eO4y6pUR1FsifKVxBDSKEIcaK2bKGs+gUogstCkoWpIIQkhpiUQ4kvUr8nHFQhUvgZXJZFIoT00Ri9H5OnpZAeMDSX6gvcnKVpCvKmeV4oGuOHf0BM8HjSyDxkYl4Yo9zHJLEsVCmflh5TkZ1eoyvU679BxqxQtZq/CsTJo2u5gJt8eKWitl6UoiG95E/JD3RYyFic0bDuJZ/TZs927H9vw4tPoYq3dewk2Xnkb2UA+ht4dEKwFE8CMLJPNMSyMoSkxxriPPQWoeI5I6UlYpzbVrK7QUX4IT98Jd/badfBbKCi5gZ7uWPD8/iu5KTT1Y3tIXd7Z56UIH1pGMt4+usswsUQ8G3Oy3SQklglQHphmDhBRWJq3YB1JJWcmm2GphIhpyXlnhrLGQLxYWpFkSVFyYK41gB4d7u1gudvDo6svxzMWbMHy8ic2Wg641RHO2h2nog1sxcmkgSxVkisJSMgnKEJMmiSQpcSxSFNNAVZPoCt+CVyx01b+EDzwUxPf8yHc/niTKj67pl0qTwxX4e4K8iw1+bsGV0bpF66OLqAYJFUfbzJHgeDeFCAKQkkMpKfCLJnJNg6gS+8IgTnTAFjQiwWI2UGyvj1J7Dg4TCo6s4uGDd8LPj+FGegzHHjuHmvsg9OocE68COQjQX/dhaQRWVUDLoUgJJs5SU0UsTcS5TsJIhJHGnKUap4bm9eySF+dXpuSKhT5VLOHmN/wYesr633DevaPe2fuJk7Un1O2jixgMm5RtKnzRWlWmhzR5hNtUGmVCVTKeNl0a2gVSlRQdowk/blCQCWZNgx0HnGVEHFvkJ4ZgzqlYnfJ67TKl4wm1M8nd0wMkUYBLdgdibwZFZSTKBWhYBCGDTh5cO0ZgEUwjQWpXGO6M01ynJDORkuACzaEECQmRkyhro0m55otcXluhb9zr4G2Dn8M9P/zvxpNw+X0z3z1Y2Rm9+kjRl7XikC+2jirDTYGd/iIKjYBMjDCr2PycegjTqIjSMEKYqBzEVchEQNNjqInKknJOTRXJzARyhb2KTRdNF3owRzwlyhKCqM5QnAYIpwFK5QxOFMDwp8gjidQXUEyGrSUo8Rhey2DDGZHaAaWWyqTHuRH7yJkpdwRNdeOi7thTJQyvyMcVb5v+tgusvHwR02AV6/ELE7moXJwbhZdTmjYLe2NQIaUxNTjJNKppE3LHI+wV6zQctCRvRqBxJJCSSF2TqcRw6h4VnB7sqk95GSTtEHBDwMmgzBPk/YTyYC4iqZFuC7REF9poDOEY2HGWEE1VkGFCa4SoihkMP4fOPlfdoSwFEoqnIbJV9lWDTS9BwxhjYpfTLeXAR53+6Uc2K3fD+9rfXzuhADC98BCuu6uAcdPCgx/4+53oYw9PJnrpTtcI3VrYh2/XkZAuFvRdtoczeG6FJt0SjHGf1bIBfVlSrTyklrIt1rJLvDK5TI2kx4UoQG02pIoYoVwYouD6SGyFUqNEWaBTYhWgVxOUayl2i+vYTtYRJzZQy9Ew+7DHMWRM0AXJ1NA5yQwoUkI3I8SZAfgqlSoz0Rar3W588P0h1XfMzW0Mzp++tkIBYOebF9C85RCcd30O8aP2uajkOupC+rIF7qkTKiLLLF4Ql8kdjymoFrjjL1MGA5XVAMfk82iOu2S2EzbCKZl+/L++8pKZYklmkKI2HqIxHbDrTmheLYlA1EhLBYTUMM8b6GIJaU6oWrtYjS+j1ptC5gpGTlletFv55WwBXaVKBd3nYjiWfmJTolow7YRG0dqDw+nGh9WYovN/+8Er8nBV/8s/+8EvYP3Gfwn1tX4cJOUPGlZwo4iUH1KlzLNM40C3RIMANQpJ1yT7BYMUmgDdGGGiSr+gUdJcQMYKi5RJUYlyLYOdhWyMSMpeTnoci+XGHolKjhAl6LMETEDR9bAQeVgY9qH6KTIb6Deq2PaWZPQCgMwh5YCDpLbJRjcEmTnYZfiJE8Sp9YXeV94+PnjnO67YwdVvCZ/HaNz8DXzvA9wZ3zH5PbUfHlUawbEoN+Ukr+WLZkcBclLUGKaqSTPyRe4zR01HbpYWxGzU5FzapMsEFksoFLMwQtacQDSO9BVjHAqlHaI13wVqOxAigp4x7G4CTDNkiopevQYPNUx2LU6mOsvJjJSqza7toexNkUWg2DGZKafId56PpPmlG1/1k5inq1cc/6o3OniTZ1F5+VEY3W3c9g+jvbbLSVCp3NNL1gwDU1mKx5RZAqO8gQwWGlqbyv5IzOsm7yUHwBMXikigW7FMVEnzWKdgqtJ8VlCHSk2JbBeqEUCbJlC8FOYsgp5ISEXDoGTzprWEC9FBirwCzCTKUSE2GhqVWjMc4nNc3R3y0KxR12rCjLI4CQsfOJXc94US7/H5h953xfm/JUMLZ97zCF53g46z9zpcqPkfH2Wtl+ZD+RMpKZQKRepxCqHElMUKMujEkJQzKXkqctOecKt4kSuTAcWaTsPGguLFNZGPdcJAYmaVENYAdyWEGwdAzgh1cCRdnsVunsWGohLIqPZ5Od+VSp4hT20uTCO2pY9dqvNItlhHrshQf2KUFv/qqPEZ+dxX/wp0FSYOvmVTIL/7bIJf8Ex8Xv3+mVevPSAHo5PRqn4j62B9HkGtRsgTwHdsSEWwmuUyIVDOpsJ5BqXjk5XOlfV6SEmtS91GHUFaQRpKpq6KOVnwdJvj3GA5Y0nI2NBCNJw23DRhczjLS5EHUlUeujbaZosSKJyECmxHkspaz+OFP75n5aHtjz/901havQ3Ab1xx7m9pB/PxXy/ihYctJC98ridKRyUW9FcsKm3dmfroVhrwhgVodoIa91jJmQd6BdHYQqJbrFYA1WBWIFnELM15zK4cS1mKZAA3jwZSBt1BHkcqKw5QLnV4KdqmRjgUOicIjEI+K5Z5alao77a4l60hZhd2NWVLE/k0an6o4699eDtczoqFi7h8+oGrkvlbKvTpLwcolE7AeOlt4OripUbjwsZCPDxhcE49u8lezxJCg6i5Y1gTnzJX0JwKFPZcGkkbnu3y1K3x3LbYyvzcGXm54mdMxRhhrDOnKtSGgobZo9qwAztPMHLKomu0sr3pIo/SMhCrKCDkstmRDXhwcsh5WPhM16v/jmtOxooqEU43MOt+7dtfKABMZxfwzk+fRvZf5tEh9+KuneR3x5bdmMHhubSg+3OWpgKXfRRmE1JLEoFmcjxWOOnn8McaAhQgy8SqIpnGvkJGIqZKgSK1JOzSmJpen1QBOa5VaTdeQTxW0kQqZFqSFktdrETb7KY+WZWIE936cltv/maj1ts8/dB9GJ1791WT+aIIBYDKSguriYdfemzS+czRg7Ij669w8kivWiPIWGIWusgMi+1kxrXZELbry6xkIIdGKiXSUDiPdCHsbKJYaYjI0XnCDcTCkgVljGo0BbmSJ4YDlrqs2TNulSdcVWJKhUVjrcKJYuVl3/tClbzfvONzzz7/+MtPAPMUUf/Rq5r1RRF66stznNxK8Tu33s/n+LqLkwmtCZHcsOi32TBihI7DvnR4LgosFaAYTbnEE1nSptKyAoISUSEaK4XJmARydO0Gj6dVcAoummNRmoygzmPWKeGKmOY1dSYq+QxpSaVLyWExHldmjhP85fps77fPvmz1/LjcADeAC5/7+FXP+qJN0r3q3y7h/EcfxtothYgSeb4cDE8O9NrSNHPhzqZSMxkz3eV5XqSAbEBlZFEqMEsUPQigyRi5AI+NGg+DGkX9hEUmoBYYiohlApUDxUGoFuS0XIAVhooexhio5RfmfeP3Pdf9w3Qium86fQY0n+Cv/+bityTniyb0oYdnUI/+MF6tPI1PZ/+9f5f9/vOjtHhs6usLviiQSswufGkoE85SUD9yqZ1UxTh22EORp1aFZ6JEcWwAnIEVYghQRg58YfE8KvBsqvGMi6woUlpjr5sn6V+NVfOde89rn62tyujU44/h8UcV/NaXr7xt8R/jRZ1Gbr/wKehHFvAG+SZ85PPtB2+4fv2tup3+M8UO7o3TfGM2Vyw9ZVgYcAMStmJnE7MgMqmBhOQKjdgOPQYxxY6JLNchU4JUBM/dIsdpIUYmdxB6D8ZB8Ndemj/qls3AajFGY/N/b9yv/uTH/8k1udHh5vv+BSI4qI4extedhnrH/JGDVj++c6qXX9k1lm6I57TgKr5TKfpqUfGJckKqC+lnOocTDbFUwLZOrphzyfNiKWjQL9VemHjFh1Rk/1Aud89UQg5kNMUvX8rxy7e66D95pW1g38ZCAaB17DXonP1bVO98CQ5nF/D8o1/EjSfudLf0Q6uTduEIFD5eWIiW15PtJcuPil6poG7zMkV9kWsy8C2X2mV71HEC/4WQjbN9pbI52M0mCytF6Th9fOypPt52h4B/WuKx2YuX65rfOfKyCjAVAotCIqxZ2DbWMO07mJ1QxPreRa3RGVnQSJ+UG6ITNZBHzGXLy+rUD8q9OKElK/cyDQOjgvnMBIcZhADG/QvXJM81v3Pk0TEASJwBgH4IY8mCjBnWVi6lacVkKLHCDJUkFJUhQeBcINYczOsZgghw4GPn4hV10Oyzzz777LPPPvvss88+++yzzz777LPPPvvs8//I/wTNTxWl3nxL/QAAAABJRU5ErkJggg=="
class IdleBeast:
    """The Beast rolls back and forth while a computation runs — a CSS animation the
    browser keeps playing while the kernel is busy (works in JupyterLite, Colab and
    locally). Call .stop() when the computation finishes."""
    _CSS = "@keyframes beastroll{from{left:1%;transform:rotate(0)}to{left:calc(99% - 54px);transform:rotate(720deg)}}"
    def __init__(self, label="computing"):
        self.label = label
        self.h = display(HTML(self._html(True)), display_id=True)
    def _img(self, anim):
        s = "position:absolute;bottom:0;animation:beastroll 2.4s ease-in-out infinite alternate" if anim else "vertical-align:middle"
        return f'<img src="data:image/png;base64,{_BEAST}" style="{s};height:54px;width:54px">'
    def _html(self, run):
        if run:
            return (f'<div style="position:relative;height:66px;max-width:440px"><style>{self._CSS}</style>'
                    f'{self._img(True)}<span style="position:absolute;bottom:6px;right:6px;font:.8rem sans-serif;color:#aaa">{self.label}…</span></div>')
        return f'<div style="font:.95rem sans-serif;color:#555">{self._img(False)}&nbsp; <b>{self.label} — done ✅</b></div>'
    def stop(self):
        if self.h is not None:                         # only with a real notebook frontend
            self.h.update(HTML(self._html(False)))

# a heated cavity: the cup cross-section, warm on one wall, cool on the other
Wb, Wt, H = 4.0, 5.0, 6.0
body = (WorkPlane().MoveTo(-Wb/2, 0).LineTo(Wb/2, 0)
        .LineTo(Wt/2, H).LineTo(-Wt/2, H).Close().Face())
body.edges.name = "insulated"
for e in body.edges:
    if   e.center[0] < -0.5: e.name = "warm"
    elif e.center[0] >  0.5: e.name = "cool"
mesh = Mesh(OCCGeometry(body, dim=2).GenerateMesh(maxh=0.4))
mesh.Curve(2)

## 1. Two fields, two symmetric operators

Velocity lives in a `VectorH1` (no-slip on every wall), temperature in an `H1`
(warm$=1$, cool$=0$). The **velocity** operator is a Stokes problem written with
a large **grad–div penalty** $\gamma$ instead of a pressure unknown — that makes
it symmetric positive definite and keeps $\nabla\!\cdot\mathbf u\approx0$. We
factorise it **once**.

In [ ]:
Ra, gamma = 1e4, 1e7
V = VectorH1(mesh, order=2, dirichlet="warm|cool|insulated")
S = H1(mesh, order=2, dirichlet="warm|cool")
u, vv = V.TnT()
T, St = S.TnT()

a_vel = BilinearForm(InnerProduct(Grad(u), Grad(vv))*dx
                     + gamma*div(u)*div(vv)*dx).Assemble()          # SPD
inv_vel = a_vel.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky")

gfu = GridFunction(V)                                              # velocity
gfT = GridFunction(S)                                             # temperature
gfT.Set(0.5 - x/Wt)                                              # smooth start …
gfT.Set(mesh.BoundaryCF({"warm": 1.0, "cool": 0.0}), BND)        # … with warm/cool fixed

## 2. The IMEX time loop

Each step: (i) solve the **velocity** from the current temperature (buoyancy on
the right); (ii) advance the **temperature** by implicit Euler — diffusion
*implicit* (symmetric), the advection $\mathbf u\!\cdot\!\nabla T$ moved to the
right-hand side (**explicit**), plus a little **streamline diffusion** for
stability. Both solves are `sparsecholesky`. We store frames to animate.

In [ ]:
M  = BilinearForm(T*St*dx).Assemble()
dt, tau, nsteps = 0.008, 0.25, 400
gfT.AddMultiDimComponent(gfT.vec)                                 # frame 0
roller = IdleBeast("convecting")
for step in range(nsteps):
    # (i) velocity from buoyancy  (symmetric solve)
    fbuo = LinearForm(Ra*gfT*vv[1]*dx).Assemble()
    gfu.vec.data = inv_vel * fbuo.vec

    # (ii) temperature: implicit diffusion (+streamline) — explicit convection
    Kdiff = BilinearForm(grad(T)*grad(St)*dx
                         + tau*(gfu*grad(T))*(gfu*grad(St))*dx).Assemble()
    mstar = M.mat.CreateMatrix()
    mstar.AsVector().data = M.mat.AsVector() + dt*Kdiff.mat.AsVector()
    inv_T = mstar.Inverse(S.FreeDofs(), inverse="sparsecholesky")
    conv = LinearForm((gfu*grad(gfT))*St*dx).Assemble()           # u·∇T, explicit
    res = (M.mat*gfT.vec - dt*conv.vec - mstar*gfT.vec).Evaluate()  # keeps Dirichlet
    gfT.vec.data += inv_T * res
    if step % 16 == 0:
        gfT.AddMultiDimComponent(gfT.vec)
roller.stop()

speed = Integrate(sqrt(gfu*gfu), mesh) / Integrate(CF(1), mesh)
print(f"reached a convecting state: mean flow speed {speed:.1f}, "
      f"∫(div u)² = {Integrate(div(gfu)**2, mesh):.0e}")
Draw(gfT, mesh, "temperature (press play)", interpolate_multidim=True, animate=True)

## 3. The convection roll

Warm fluid climbs the hot wall, drifts across the top, cools and sinks down the
cold wall — a single slow **convection roll**. The arrows make it plain.

In [ ]:
print("rises at the warm wall  (u_y > 0):", round(gfu(mesh(-1.5, 3))[1], 1))
print("sinks at the cool wall  (u_y < 0):", round(gfu(mesh( 1.5, 3))[1], 1))
Draw(gfu, mesh, "velocity — the convection roll", vectors={"grid_size": 26})

## Where the ☕ has taken us

From a cold chocolate bar to a convecting cup, the same handful of ideas kept
returning — geometry, coefficient functions, spaces, weak forms, solvers — only
the physics grew richer. One last flourish remains before the expedition proper:
a nonlinear PDE on the **skin of the beast** itself — where the beast finally gets
its stripes.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("13-elasticity", "13 · Making a chocolate bar bend 🍫")
    _next = ("15-turing-patterns", "15 · How the beast got its stripes 🌈")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))